# 05_phase3_pseudobulk_DE.ipynb
Phase 3 — Pseudobulk Differential Expression

**Design (see `docs/methods_phase3.md` for full reasoning):**
- GSE114725: Tumour vs Normal, 5 viable cell types (T cells, CD8/Effector T cells, NK/Cytotoxic T cells, B cells, Macrophages)
- GSE176078: Pairwise subtype comparisons (ER+ vs TNBC, ER+ vs HER2+, TNBC vs HER2+), 12 viable cell types
- Pseudobulk: raw integer counts aggregated (summed) per patient/sample within each cell type, tested with PyDESeq2 (negative binomial model — requires raw counts, NOT the log-normalised data used for clustering)
- FDR correction (Benjamini-Hochberg) applied via DESeq2's built-in padj

In [1]:
# Cell 1 — Imports and paths
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_pseudobulk_de"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_pseudobulk_de"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Setup complete


In [2]:
# Cell 1b — Verify correct input files before running any DE
# Confirms: genuine raw integer counts, MT/VDJ genes removed, cell
# counts match the validated Phase 2 annotation.
vdj_prefixes = ("IGHV", "IGLV", "IGKV", "TRAV", "TRBV", "TRGV", "TRDV",
                 "IGHD", "IGHJ", "IGLJ", "IGKJ", "TRAC", "TRBC", "TRGC", "TRDC",
                 "IGHA", "IGHE", "IGHG", "IGHM", "IGLC", "IGKC")

for name, path in [
    ("GSE114725", PROCESSED_DIR / "GSE114725_phase1_v2_clean_rawcounts.h5ad"),
    ("GSE176078", PROCESSED_DIR / "GSE176078_phase1_v2_clean_rawcounts.h5ad"),
]:
    check = sc.read_h5ad(path, backed="r")
    mt_genes = [g for g in check.var_names if g.startswith("MT-")]
    vdj_genes = [g for g in check.var_names if g.startswith(vdj_prefixes)]
    print(f"{name}: {check.n_obs} cells x {check.n_vars} genes, "
          f"MT genes: {len(mt_genes)}, VDJ genes: {len(vdj_genes)}")
    if mt_genes or vdj_genes:
        raise ValueError(f"{name} checkpoint still contains MT/VDJ genes — wrong file or fix incomplete")
    del check

print("\nBoth checkpoints verified clean — safe to proceed with DE")

GSE114725: 44662 cells x 14800 genes, MT genes: 0, VDJ genes: 0
GSE176078: 91425 cells x 27343 genes, MT genes: 0, VDJ genes: 0

Both checkpoints verified clean — safe to proceed with DE


In [3]:
def load_raw_with_annotation(raw_path, annotated_path, dataset_name):
    print(f"Loading {dataset_name}...")
    raw = sc.read_h5ad(raw_path)
    annotated = sc.read_h5ad(annotated_path, backed="r")

    sample_vals = raw.X[:100].toarray() if hasattr(raw.X, "toarray") else raw.X[:100]
    is_integer_like = np.allclose(sample_vals, np.round(sample_vals))
    print(f"  Raw counts appear to be integers: {is_integer_like}")
    if not is_integer_like:
        raise ValueError(
            f"{dataset_name} raw.X does NOT look like integer counts — "
            f"wrong file loaded, or already normalised. STOP and check "
            f"before running PyDESeq2 on this."
        )

    qc_passed_barcodes = annotated.obs_names
    raw_qc = raw[raw.obs_names.isin(qc_passed_barcodes)].copy()
    print(f"  Raw: {raw.n_obs} cells (pre-QC) -> {raw_qc.n_obs} cells (post-QC, matches Phase 2)")
    assert raw_qc.n_obs == annotated.n_obs, (
        f"Cell count mismatch after QC subsetting: {raw_qc.n_obs} vs {annotated.n_obs}. "
        f"Barcode overlap may be incomplete — check before proceeding."
    )

    meta_cols = [c for c in annotated.obs.columns]
    raw_qc.obs = raw_qc.obs.join(annotated.obs[meta_cols], rsuffix="_annotated")

    del raw
    gc.collect()
    return raw_qc

# CHANGED: load from the new clean checkpoints (MT/VDJ already removed),
# not the fully-unfiltered original raw files
adata1_raw = load_raw_with_annotation(
    PROCESSED_DIR / "GSE114725_phase1_v2_clean_rawcounts.h5ad",
    PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad",
    "GSE114725"
)
adata2_raw = load_raw_with_annotation(
    PROCESSED_DIR / "GSE176078_phase1_v2_clean_rawcounts.h5ad",
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad",
    "GSE176078"
)

print(f"\nGSE114725: {adata1_raw.n_obs} cells x {adata1_raw.n_vars} genes")
print(f"GSE176078: {adata2_raw.n_obs} cells x {adata2_raw.n_vars} genes")

Loading GSE114725...
  Raw counts appear to be integers: True
  Raw: 44662 cells (pre-QC) -> 44662 cells (post-QC, matches Phase 2)
Loading GSE176078...
  Raw counts appear to be integers: True
  Raw: 91425 cells (pre-QC) -> 91425 cells (post-QC, matches Phase 2)

GSE114725: 44662 cells x 14800 genes
GSE176078: 91425 cells x 27343 genes


In [4]:
# Cell 3 — Pseudobulk aggregation function
def build_pseudobulk(adata_raw, sample_col, cell_type, min_cells=10):
    """
    Returns (counts_df, sample_metadata_df) for one cell type.
    counts_df: genes x samples (raw summed counts)
    sample_metadata_df: one row per sample, indexed the same way
    """
    subset = adata_raw[adata_raw.obs["cell_type"] == cell_type]

    pseudobulk_samples = []
    sample_ids = []
    cell_counts_per_sample = []

    for sample_id in subset.obs[sample_col].unique():
        sample_mask = (subset.obs[sample_col] == sample_id).values
        n_cells = sample_mask.sum()
        if n_cells < min_cells:
            continue  # matches the viability check already done
        X_sample = subset.X[sample_mask]
        summed = np.asarray(X_sample.sum(axis=0)).flatten()
        pseudobulk_samples.append(summed)
        sample_ids.append(sample_id)
        cell_counts_per_sample.append(n_cells)

    counts_df = pd.DataFrame(
        pseudobulk_samples, index=sample_ids, columns=subset.var_names
    ).T  # genes x samples, as PyDESeq2 expects

    meta_df = pd.DataFrame({
        sample_col: sample_ids,
        "n_cells": cell_counts_per_sample
    }, index=sample_ids)

    return counts_df, meta_df

print("Pseudobulk aggregation function ready")

Pseudobulk aggregation function ready


In [5]:
# ----------------------------
# Cell 4 — PyDESeq2 runner for one comparison
# FIX 1: prints sample counts BEFORE attempting the fit (traceable failures)
# FIX 2: meta_sub[group_col] cast to plain string after filtering — pandas
# Categorical dtype retains ALL original category levels even after rows
# are filtered out, causing PyDESeq2 to build a dummy design-matrix column
# for an absent category, producing a singular (non-invertible) matrix.
# FIX 3: catches fit failures gracefully instead of crashing the whole loop.
# ----------------------------
def run_pydeseq2(counts_df, meta_df, group_col, group_a, group_b,
                  cell_type, comparison_name, dataset_name, min_genes_expressed=10):
    meta_sub = meta_df[meta_df[group_col].isin([group_a, group_b])].copy()
    meta_sub[group_col] = meta_sub[group_col].astype(str)  # drop unused categorical levels

    counts_sub = counts_df[meta_sub.index]
    gene_filter = (counts_sub > 0).sum(axis=1) >= min_genes_expressed
    counts_sub = counts_sub[gene_filter]

    n_a = (meta_sub[group_col] == group_a).sum()
    n_b = (meta_sub[group_col] == group_b).sum()

    print(f"  Attempting {dataset_name} | {cell_type} | {comparison_name}: "
          f"{n_a} {group_a} / {n_b} {group_b} samples, {len(counts_sub)} genes after filtering")

    if n_a < 2 or n_b < 2:
        print(f"    SKIPPED — need >=2 samples per group (got {n_a}/{n_b})")
        return None

    counts_for_deseq = counts_sub.T.astype(int)

    try:
        dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
                            design_factors=group_col, refit_cooks=True, quiet=True)
        dds.deseq2()
        ds = DeseqStats(dds, contrast=[group_col, group_a, group_b], quiet=True)
        ds.summary()
        results = ds.results_df.copy().sort_values("padj")
        n_sig = (results["padj"] < 0.05).sum()
        print(f"    SUCCESS — {len(results)} genes tested, {n_sig} significant (padj<0.05)")
        return results
    except Exception as e:
        print(f"    FAILED — {type(e).__name__}: {e}")
        return None

print("PyDESeq2 runner ready (dtype fix + diagnostics)")

PyDESeq2 runner ready (dtype fix + diagnostics)


In [6]:
# ----------------------------
# Cell 5 — GSE114725: Tumour vs Normal, 5 viable cell types
# ----------------------------
viable_cell_types_1 = [
    "T cells", "CD8/Effector T cells", "NK/Cytotoxic T cells",
    "B cells", "Macrophages"
]

all_results_1 = {}

for ct in viable_cell_types_1:
    counts_df, meta_df = build_pseudobulk(
        adata1_raw, sample_col="patient", cell_type=ct, min_cells=10
    )
    # restrict metadata to tissue == TUMOR or NORMAL for this comparison
    tissue_lookup = adata1_raw.obs.drop_duplicates("patient").set_index("patient")
    # patients can have multiple tissues — need per-(patient,tissue) pseudobulk,
    # not per-patient alone, since a patient contributes separately to each tissue
    subset = adata1_raw[adata1_raw.obs["cell_type"] == ct]
    subset = subset[subset.obs["tissue"].isin(["TUMOR", "NORMAL"])]
    subset.obs["sample_id"] = subset.obs["patient"].astype(str) + "_" + subset.obs["tissue"].astype(str)

    pseudobulk_samples, sample_ids, tissue_labels, n_cells_list = [], [], [], []
    for sid in subset.obs["sample_id"].unique():
        mask = (subset.obs["sample_id"] == sid).values
        n_cells = mask.sum()
        if n_cells < 10:
            continue
        summed = np.asarray(subset.X[mask].sum(axis=0)).flatten()
        pseudobulk_samples.append(summed)
        sample_ids.append(sid)
        tissue_labels.append(subset.obs.loc[mask, "tissue"].iloc[0])
        n_cells_list.append(n_cells)

    counts_df = pd.DataFrame(pseudobulk_samples, index=sample_ids, columns=subset.var_names).T
    meta_df = pd.DataFrame({"tissue": tissue_labels, "n_cells": n_cells_list}, index=sample_ids)

    results = run_pydeseq2(
        counts_df, meta_df, group_col="tissue", group_a="TUMOR", group_b="NORMAL",
        cell_type=ct, comparison_name="Tumor_vs_Normal", dataset_name="GSE114725"
    )
    if results is not None:
        all_results_1[ct] = results
        safe_ct = ct.replace("/", "_").replace(" ", "_")
        results.to_csv(RESULTS_DIR / f"GSE114725_DE_{safe_ct}_tumor_vs_normal.csv")
        results[results["padj"] < 0.05].to_csv(
            RESULTS_DIR / f"GSE114725_DE_{safe_ct}_tumor_vs_normal_significant.csv")

print("\nGSE114725 pseudobulk DE complete")

C:\Users\annam\AppData\Local\Temp\ipykernel_18640\3112899855.py:21: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  subset.obs["sample_id"] = subset.obs["patient"].astype(str) + "_" + subset.obs["tissue"].astype(str)


  Attempting GSE114725 | T cells | Tumor_vs_Normal: 8 TUMOR / 3 NORMAL samples, 6791 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\1875344533.py:32: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.05 seconds.

Fitting MAP dispersions...
... done in 0.06 seconds.

Fitting LFCs...
... done in 0.06 seconds.



    SUCCESS — 6791 genes tested, 1 significant (padj<0.05)


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\3112899855.py:21: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  subset.obs["sample_id"] = subset.obs["patient"].astype(str) + "_" + subset.obs["tissue"].astype(str)


...

    SUCCESS — 1602 genes tested, 1 significant (padj<0.05)


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\3112899855.py:21: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  subset.obs["sample_id"] = subset.obs["patient"].astype(str) + "_" + subset.obs["tissue"].astype(str)


  Attempting GSE114725 | Macrophages | Tumor_vs_Normal: 8 TUMOR / 4 NORMAL samples, 9267 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\1875344533.py:32: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.17 seconds.

Fitting MAP dispersions...
... done in 0.18 seconds.

Fitting LFCs...
... done in 0.17 seconds.



    SUCCESS — 9267 genes tested, 12 significant (padj<0.05)

GSE114725 pseudobulk DE complete


In [7]:
# ----------------------------
# Cell 6 — GSE176078: pairwise subtype comparisons, 12 viable cell types
# ----------------------------
viable_cell_types_2 = [
    "Endothelial cells", "CAFs", "PVL", "B cells", "CD8 T cells", "NK cells",
    "Memory T cells", "T cells", "Cycling epithelial", "Macrophages",
    "Epithelial (ambiguous)", "Luminal epithelial"
]

pairwise_comparisons = [
    ("TNBC", "ER+"), ("HER2+", "ER+"), ("TNBC", "HER2+")
]

all_results_2 = {}

for ct in viable_cell_types_2:
    counts_df, meta_df = build_pseudobulk(
        adata2_raw, sample_col="orig.ident", cell_type=ct, min_cells=10
    )
    # attach subtype metadata per sample
    subtype_lookup = adata2_raw.obs.drop_duplicates("orig.ident").set_index("orig.ident")["subtype"]
    meta_df["subtype"] = meta_df["orig.ident"].map(subtype_lookup)

    for group_a, group_b in pairwise_comparisons:
        comparison_name = f"{group_a}_vs_{group_b}"
        results = run_pydeseq2(
            counts_df, meta_df, group_col="subtype", group_a=group_a, group_b=group_b,
            cell_type=ct, comparison_name=comparison_name, dataset_name="GSE176078"
        )
        if results is not None:
            all_results_2[(ct, comparison_name)] = results
            safe_ct = ct.replace("/", "_").replace(" ", "_").replace("(", "").replace(")", "")
            results.to_csv(RESULTS_DIR / f"GSE176078_DE_{safe_ct}_{comparison_name}.csv")
            results[results["padj"] < 0.05].to_csv(
                RESULTS_DIR / f"GSE176078_DE_{safe_ct}_{comparison_name}_significant.csv")

print("\nGSE176078 pseudobulk DE complete")

  Attempting GSE176078 | Endothelial cells | TNBC_vs_ER+: 9 TNBC / 11 ER+ samples, 13262 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\1875344533.py:32: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.11 seconds.

Fitting MAP dispersions...
... done in 0.10 seconds.

Fitting LFCs...
... done in 0.10 seconds.



    SUCCESS — 13262 genes tested, 1 significant (padj<0.05)
  Attempting GSE176078 | Endothelial cells | HER2+_vs_ER+: 5 HER2+ / 11 ER+ samples, 12502 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\1875344533.py:32: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.05 seconds.

Fitting MAP dispersions...
... done in 0.05 seconds.

Fitting LFCs...
... done in 0.07 seconds.



    SUCCESS — 12502 genes tested, 0 significant (padj<0.05)
  Attempting GSE176078 | Endothelial cells | TNBC_vs_HER2+: 9 TNBC / 5 HER2+ samples, 11573 genes after filtering


...

    SUCCESS — 15766 genes tested, 1603 significant (padj<0.05)
  Attempting GSE176078 | Luminal epithelial | HER2+_vs_ER+: 4 HER2+ / 9 ER+ samples, 14353 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\1875344533.py:32: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.44 seconds.

Fitting MAP dispersions...
... done in 0.48 seconds.

Fitting LFCs...
... done in 0.46 seconds.



    SUCCESS — 14353 genes tested, 562 significant (padj<0.05)
  Attempting GSE176078 | Luminal epithelial | TNBC_vs_HER2+: 9 TNBC / 4 HER2+ samples, 12112 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_18640\1875344533.py:32: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.45 seconds.

Fitting MAP dispersions...
... done in 0.44 seconds.

Fitting LFCs...
... done in 0.46 seconds.



    SUCCESS — 12112 genes tested, 17 significant (padj<0.05)

GSE176078 pseudobulk DE complete


In [8]:
# ----------------------------
# Cell 7 — Summary table across all comparisons
# ----------------------------
summary_rows = []
for ct, results in all_results_1.items():
    summary_rows.append({
        "dataset": "GSE114725", "cell_type": ct, "comparison": "Tumor_vs_Normal",
        "n_genes_tested": len(results), "n_significant_padj05": (results["padj"] < 0.05).sum()
    })
for (ct, comp), results in all_results_2.items():
    summary_rows.append({
        "dataset": "GSE176078", "cell_type": ct, "comparison": comp,
        "n_genes_tested": len(results), "n_significant_padj05": (results["padj"] < 0.05).sum()
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / "phase3_DE_summary_all_comparisons.csv", index=False)
print(summary_df.to_string(index=False))

  dataset              cell_type      comparison  n_genes_tested  n_significant_padj05
GSE114725                T cells Tumor_vs_Normal            6791                     1
GSE114725   CD8/Effector T cells Tumor_vs_Normal            7324                     0
GSE114725   NK/Cytotoxic T cells Tumor_vs_Normal            3926                     1
GSE114725                B cells Tumor_vs_Normal            1602                     1
GSE114725            Macrophages Tumor_vs_Normal            9267                    12
GSE176078      Endothelial cells     TNBC_vs_ER+           13262                     1
GSE176078      Endothelial cells    HER2+_vs_ER+           12502                     0
GSE176078      Endothelial cells   TNBC_vs_HER2+           11573                     0
GSE176078                   CAFs     TNBC_vs_ER+           13706                    11
GSE176078                   CAFs    HER2+_vs_ER+           11946                     0
GSE176078                   CAFs   TNBC_vs_

## Figures — Volcano plots

In [9]:
# ----------------------------
# Cell 8 — Volcano plots for all DE comparisons
# Standard DE visualisation: log2FoldChange (x) vs -log10(padj) (y).
# Significant genes (padj<0.05) highlighted; top genes by padj labelled.
# Saved individually per comparison so specific ones can be pulled into
# the thesis as needed, rather than one crowded combined figure.
# ----------------------------
import matplotlib.pyplot as plt
import numpy as np

def plot_volcano(results_df, title, save_path, n_label=8, padj_thresh=0.05, lfc_thresh=1.0):
    df = results_df.copy()
    df = df.dropna(subset=["log2FoldChange", "padj"])
    df["neg_log10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))

    sig_up = (df["padj"] < padj_thresh) & (df["log2FoldChange"] > lfc_thresh)
    sig_down = (df["padj"] < padj_thresh) & (df["log2FoldChange"] < -lfc_thresh)
    not_sig = ~(sig_up | sig_down)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(df.loc[not_sig, "log2FoldChange"], df.loc[not_sig, "neg_log10_padj"],
               c="lightgrey", s=10, alpha=0.5, label="Not significant")
    ax.scatter(df.loc[sig_up, "log2FoldChange"], df.loc[sig_up, "neg_log10_padj"],
               c="firebrick", s=15, alpha=0.7, label="Up (padj<0.05)")
    ax.scatter(df.loc[sig_down, "log2FoldChange"], df.loc[sig_down, "neg_log10_padj"],
               c="steelblue", s=15, alpha=0.7, label="Down (padj<0.05)")

    top_genes = df[sig_up | sig_down].sort_values("padj").head(n_label)
    for gene, row in top_genes.iterrows():
        ax.annotate(gene, (row["log2FoldChange"], row["neg_log10_padj"]),
                    fontsize=8, ha="center", va="bottom",
                    xytext=(0, 3), textcoords="offset points")

    ax.axhline(-np.log10(padj_thresh), color="grey", linestyle="--", linewidth=0.8)
    ax.axvline(lfc_thresh, color="grey", linestyle="--", linewidth=0.8)
    ax.axvline(-lfc_thresh, color="grey", linestyle="--", linewidth=0.8)
    ax.set_xlabel("log2 Fold Change")
    ax.set_ylabel("-log10(adjusted p-value)")
    ax.set_title(title, fontsize=11)
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

# GSE114725
for ct, results in all_results_1.items():
    safe_ct = ct.replace("/", "_").replace(" ", "_")
    plot_volcano(
        results, f"GSE114725 — {ct}\nTumour vs Normal",
        FIGURE_DIR / f"GSE114725_volcano_{safe_ct}_tumor_vs_normal.png"
    )

# GSE176078
for (ct, comp), results in all_results_2.items():
    safe_ct = ct.replace("/", "_").replace(" ", "_").replace("(", "").replace(")", "")
    plot_volcano(
        results, f"GSE176078 — {ct}\n{comp.replace(chr(95), chr(32))}",
        FIGURE_DIR / f"GSE176078_volcano_{safe_ct}_{comp}.png"
    )

print(f"Volcano plots saved: {len(all_results_1) + len(all_results_2)} figures")

Volcano plots saved: 41 figures


In [10]:
sig_files = list((PROJECT_DIR / "results" / "phase3_pseudobulk_de").glob("*_significant.csv"))
found_any = False
for f in sig_files:
    df = pd.read_csv(f, index_col=0)
    mt_hits = [g for g in df.index if str(g).startswith("MT-")]
    vdj_hits = [g for g in df.index if str(g).startswith(vdj_prefixes)]
    if mt_hits or vdj_hits:
        print(f"{f.name}: MT hits={mt_hits}, VDJ hits={vdj_hits}")
        found_any = True

if not found_any:
    print("No MT or VDJ genes found in any significant DE result — fix confirmed working.")

No MT or VDJ genes found in any significant DE result — fix confirmed working.


In [11]:
# Quick check: macrophage tumour-vs-normal result still intact
mac_result = pd.read_csv(RESULTS_DIR / "GSE114725_DE_Macrophages_tumor_vs_normal_significant.csv", index_col=0)
print(f"GSE114725 Macrophages tumour-vs-normal: {len(mac_result)} significant genes")
print(mac_result.index.tolist())

GSE114725 Macrophages tumour-vs-normal: 12 significant genes
['HSPA1A', 'FN1', 'PPIF', 'HSPA1B', 'HEATR5B', 'FAM110B', 'C6ORF226', 'UTP14A', 'CEP290', 'ZNF212', 'MEGF8', 'TMF1']
